# Kết quả GraphX và GraphFrames

Đọc lần benchmark upstream thành công gần nhất, loại warmup và kiểm tra correctness.
Không chạy Spark hoặc sửa dữ liệu.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'upstream.json').is_file())
latest = json.loads((ROOT / 'reports/latest_run.json').read_text())
RUN = ROOT / latest['run_dir']
assert (RUN / 'SUCCESS').is_file(), 'Benchmark chưa hoàn tất'
manifest = json.loads((RUN / 'input_manifest.json').read_text())
assert manifest['edge_view'] == 'transactions'
schedule = pd.read_csv(RUN / 'schedule.csv')
checks = pd.read_csv(RUN / 'correctness.csv')
assert len(checks) == 2 * len(schedule) and checks['passed'].all()
assert checks.groupby('run')['kind'].agg(set).apply(lambda x: x == {'incoming', 'pagerank'}).all()
assert set(checks['run']) == set(schedule['run'])
timing = pd.read_csv(RUN / 'phase_timings.csv').merge(schedule, on='run', validate='many_to_one')
measured = timing.loc[~timing['warmup']].copy()
assert set(measured['framework']) == {'GraphX', 'GraphFrames'}
print(f"Input: {manifest['vertices']:,} vertices; {manifest['edges']:,} transaction edges")
print(f"Upstream commit: {manifest['upstream_commit']}")
print(f"Run: {RUN}")

## Thời gian thực thi

`build` đo tạo representation và partition; `aggregate` đo gửi và cộng USD amount theo tài khoản nhận.
`pr_total` gồm `pr_setup` và mọi `pr_iteration`, bao gồm checkpoint.
`total` là tổng các phase được đo, không bao gồm đọc input, ID mapping, reference hoặc kiểm tra correctness.

In [ ]:
columns = ['run', 'framework', 'repeat']
phase = measured.groupby(columns + ['phase'], as_index=False)['wall_ms'].sum()
pr = phase.loc[phase.phase.isin(['pr_setup', 'pr_iteration'])].groupby(columns, as_index=False)['wall_ms'].sum().assign(phase='pr_total')
total = phase.groupby(columns, as_index=False)['wall_ms'].sum().assign(phase='total')
results = pd.concat([phase, pr, total], ignore_index=True)
summary = results.groupby(['phase', 'framework'])['wall_ms'].agg(median='median', minimum='min', maximum='max')
display(summary.round(2))
plot = summary['median'].unstack('framework').reindex(['build', 'aggregate', 'pr_total', 'total']) / 1000
ax = plot.plot.bar(rot=0, figsize=(9, 4), ylabel='Giây (median)', title='Cùng graph giao dịch upstream')
ax.figure.tight_layout()
plt.show()
analysis_dir = RUN / 'analysis'
analysis_dir.mkdir(exist_ok=True)
summary.to_csv(analysis_dir / 'timings.csv')
ax.figure.savefig(analysis_dir / 'timings.png', dpi=160)

## Giới hạn diễn giải

So sánh cùng công thức PageRank, số vòng, input, partition count và cache policy.
GraphX dùng EdgePartition2D; GraphFrames repartition theo src, AQE và broadcast join tắt.
Đây là benchmark các phép tính tương đương qua GraphX và GraphFrames aggregateMessages, không phải so sánh PageRank built-in.
Kết quả local không đo network giữa các máy và không đại diện mọi cấu hình của hai framework.
Cache bytes là kích thước block Spark lưu, không phải tổng RAM; event log và execution plan nằm trong thư mục run để phân tích sâu hơn.
Không dùng kết quả từ edges_agg để kết luận tốc độ trên graph giao dịch.